# 🐑 Sheep Skin Disease Detector — Dedicated Model

Part of your SIH multi-species project: this notebook is **sheep-only**, matching the cow and goat notebooks'
per-species architecture. Trains on **`sheepgoat_dataset.zip`** (Contagious Ecthyma / Orf disease, bounding-box
crops) plus **`HealthySheep_dataset.zip`** (whole-image Healthy class), and gives you a `.keras` file + a symptom
`.joblib` file for your website's backend.

### ⚠️ Root cause of the "real photos misclassified as Healthy" bug, and the actual fix
Earlier versions of this notebook trained *only* on tightly-cropped bounding boxes — and the Orf boxes in this
dataset cover only **~6-10% of the source photo's area on average**. Training exclusively on tight crops means
the lesion texture fills the *entire* 300×300 training image every time, so the model effectively learned "disease
= the whole frame is crusty texture." A real uploaded photo (whole face/body, lesion occupying a small natural
part of the frame) never matched that learned pattern, so the model defaulted to "Healthy" even on genuinely
diseased animals — this is why test accuracy looked excellent (99%+) while real-world photos kept failing: the
test set was built the same tightly-cropped way, so it wasn't actually representative of real usage.
**The fix (Section 4 below):** for every disease-positive source photo, the **whole, uncropped image** is now
added as an additional training example alongside the tight crop — so the model learns to recognize the lesion
both up close *and* at the realistic scale it actually appears at in a normal photo.

### ⚠️ One more honest data caveat
`sheepgoat_dataset.zip`'s filename and Roboflow README describe it as covering **both sheep and goats** — there is
no species column in its annotations to cleanly separate the two. Contagious Ecthyma (Orf) is the same virus in
both species and looks visually similar on the skin either way, so this is the best available data for a sheep
Orf detector, but a small fraction of its training images may technically be goat photos rather than sheep. This
is unlikely to hurt Orf-detection accuracy (the lesion appearance is what matters, not the surrounding animal),
but it's worth knowing for your project documentation/report.

**Final classes for this model (2):**
1. `Contagious_Ecthyma_Orf`
2. `Normal_Healthy_Skin`

**Output format matches the cow and goat notebooks:** one direct final verdict (condition + confidence, via
Test-Time Augmentation), no percentage breakdown for photo mode; symptom mode keeps its full breakdown.


## 0. GPU check

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print("GPU found:", gpus if gpus else "⚠️ NO GPU — go to Runtime → Change runtime type → GPU (T4) and restart")


## 1. Install & import everything

In [ ]:
!pip install -q gradio scikit-learn seaborn

import os, glob, shutil, json, random, zipfile, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier
import joblib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)


## 2. Load `sheepgoat_dataset.zip` and `HealthySheep_dataset.zip` from Google Drive

Set `DRIVE_FOLDER` if your zips aren't directly in **My Drive**. Filename matching is case-insensitive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# <<< EDIT THIS if your zips are inside a subfolder of My Drive >>>
DRIVE_FOLDER = '/content/drive/MyDrive'

RAW_DIR = '/content/data/raw_sheep'
os.makedirs(RAW_DIR, exist_ok=True)

EXPECTED_NAMES = {
    'sheepgoat':    'sheepgoat_dataset.zip',
    'healthysheep': 'HealthySheep_dataset.zip',
}

def find_case_insensitive(folder, filename):
    target = filename.lower()
    try:
        for f in os.listdir(folder):
            if f.lower() == target:
                return os.path.join(folder, f)
    except FileNotFoundError:
        pass
    return os.path.join(folder, filename)

def check_zip(path, min_kb=10):
    if not os.path.exists(path):
        return False, "not found in that Drive folder"
    size_kb = os.path.getsize(path) / 1024
    if size_kb < min_kb:
        return False, f"only {size_kb:.0f} KB — likely an incomplete upload to Drive"
    if not zipfile.is_zipfile(path):
        return False, f"{size_kb:.0f} KB but not a valid zip file (corrupted)"
    return True, f"OK ({size_kb:.0f} KB)"

zip_paths = {}
print("Checking Drive for expected files:")
for key, fname in EXPECTED_NAMES.items():
    path = find_case_insensitive(DRIVE_FOLDER, fname)
    ok, msg = check_zip(path)
    print(f"  {key:14s} {path:60s} -> {msg}")
    if ok:
        zip_paths[key] = path

missing = set(EXPECTED_NAMES) - set(zip_paths)
assert not missing, f"Missing/invalid in Drive: {missing}. Check DRIVE_FOLDER and filenames."

for key, zp in zip_paths.items():
    out_dir = os.path.join(RAW_DIR, key)
    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir, exist_ok=True)
    with zipfile.ZipFile(zp, 'r') as z:
        z.extractall(out_dir)
    print(f"Extracted {key} -> {out_dir}")


## 3. Auto-locate the annotation CSV and the Healthy Sheep image folder

In [ ]:
IMG_EXTS = ('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG')

sheepgoat_csvs = glob.glob(os.path.join(RAW_DIR, 'sheepgoat', '**', '_annotations.csv'), recursive=True)
print("Sheep&Goat annotation files found:")
for c in sheepgoat_csvs:
    print(" ", c)

healthysheep_root = RAW_DIR + '/healthysheep'
all_hs_dirs = glob.glob(os.path.join(healthysheep_root, '**/'), recursive=True) + [healthysheep_root + '/']
healthysheep_dir = max(all_hs_dirs, key=lambda d: len([f for f in glob.glob(os.path.join(d, '*')) if f.lower().endswith(IMG_EXTS)]))
n_hs = len([f for f in glob.glob(os.path.join(healthysheep_dir, '*')) if f.lower().endswith(IMG_EXTS)])
print(f"\nHealthy Sheep dir: {healthysheep_dir} -> {n_hs} files")


## 4. Build the unified image dataset

Orf disease images are bounding-box exports → every lesion box is **cropped out** as its own training image
(teaches close-up lesion texture), **and** the **whole uncropped source photo** is added as a second training
example for that same disease (teaches realistic in-context scale — see the root-cause note above). Healthy
images are plain whole-image files → copied directly (re-encoded as clean RGB JPEG).

In [ ]:
UNIFIED_DIR = '/content/data/unified_sheep'
if os.path.exists(UNIFIED_DIR):
    shutil.rmtree(UNIFIED_DIR)

CLASS_MAP = {
    'ORF Disease': 'Contagious_Ecthyma_Orf',
}

def ensure_dir(d):
    os.makedirs(d, exist_ok=True)
    return d

def copy_whole_images(src_dir, unified_class, prefix='img'):
    out_dir = ensure_dir(os.path.join(UNIFIED_DIR, unified_class))
    n = 0
    for f in glob.glob(os.path.join(src_dir, '*')):
        if not f.lower().endswith(IMG_EXTS + ('.gif',)):
            continue
        try:
            img = Image.open(f)
            if getattr(img, 'is_animated', False):
                img.seek(0)
            img = img.convert('RGB')
            img.save(os.path.join(out_dir, f"{prefix}_{n:05d}.jpg"), quality=95)
            n += 1
        except Exception:
            continue
    return n

def crop_bboxes_from_csv(csv_path, prefix, pad_frac=0.08):
    img_dir = os.path.dirname(csv_path)
    df = pd.read_csv(csv_path)
    counts = {}
    for i, row in df.iterrows():
        cls = str(row['class']).strip()
        if cls not in CLASS_MAP:
            continue
        unified_class = CLASS_MAP[cls]
        img_path = os.path.join(img_dir, row['filename'])
        if not os.path.exists(img_path):
            continue
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception:
            continue
        w, h = img.size
        xmin, ymin, xmax, ymax = row['xmin'], row['ymin'], row['xmax'], row['ymax']
        bw, bh = xmax - xmin, ymax - ymin
        pad_x, pad_y = bw * pad_frac, bh * pad_frac
        xmin = max(0, int(xmin - pad_x)); ymin = max(0, int(ymin - pad_y))
        xmax = min(w, int(xmax + pad_x)); ymax = min(h, int(ymax + pad_y))
        if xmax <= xmin or ymax <= ymin:
            continue
        crop = img.crop((xmin, ymin, xmax, ymax))
        out_dir = ensure_dir(os.path.join(UNIFIED_DIR, unified_class))
        counts[unified_class] = counts.get(unified_class, 0) + 1
        crop.save(os.path.join(out_dir, f"{prefix}_{i:05d}.jpg"), quality=95)
    return counts

def copy_whole_positive_images(csv_path, prefix='wholeimg'):
    '''CRITICAL for real-world accuracy: also copies each SOURCE image UNCROPPED (whole face/body) for every
    photo that has at least one disease annotation. The Orf bounding boxes in this dataset cover only ~6-10%
    of the source image on average — if training only ever sees TIGHT crops (lesion filling 100% of the
    300x300 frame), the model learns "disease = frame full of crusty texture" and fails on a real uploaded
    photo, where the lesion is naturally a small part of a larger face/body frame. Adding the whole image as
    an additional positive example teaches the model to recognize the lesion at realistic, in-context scale.'''
    img_dir = os.path.dirname(csv_path)
    df = pd.read_csv(csv_path)
    df = df[df['class'].isin(CLASS_MAP.keys())]
    counts = {}
    for i, fname in enumerate(df['filename'].unique()):
        classes_here = df.loc[df['filename'] == fname, 'class'].unique()
        img_path = os.path.join(img_dir, fname)
        if not os.path.exists(img_path):
            continue
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception:
            continue
        for cls in classes_here:
            unified_class = CLASS_MAP[cls]
            out_dir = ensure_dir(os.path.join(UNIFIED_DIR, unified_class))
            img.save(os.path.join(out_dir, f"{prefix}_{i:05d}.jpg"), quality=95)
            counts[unified_class] = counts.get(unified_class, 0) + 1
    return counts

sg_totals = {}
for csv_path in sheepgoat_csvs:
    counts = crop_bboxes_from_csv(csv_path, prefix='sheep')
    for k, v in counts.items():
        sg_totals[k] = sg_totals.get(k, 0) + v
print("Disease crops (tight, lesion fills frame):", sg_totals)

sg_whole_totals = {}
for csv_path in sheepgoat_csvs:
    counts = copy_whole_positive_images(csv_path, prefix='sheep_whole')
    for k, v in counts.items():
        sg_whole_totals[k] = sg_whole_totals.get(k, 0) + v
print("Disease whole-images (realistic scale, uncropped):", sg_whole_totals)

n_healthy = copy_whole_images(healthysheep_dir, 'Normal_Healthy_Skin', prefix='healthy')
print(f"Normal_Healthy_Skin: {n_healthy} images")

print("\n=== FINAL CLASS COUNTS ===")
class_names_unified = sorted(os.listdir(UNIFIED_DIR))
for c in class_names_unified:
    n = len(glob.glob(os.path.join(UNIFIED_DIR, c, '*')))
    print(f"  {c:25s} {n:5d} images")


## 5. Sanity-check a few samples per class

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for row, c in enumerate(class_names_unified):
    files_c = glob.glob(os.path.join(UNIFIED_DIR, c, '*'))
    sample = random.sample(files_c, min(5, len(files_c)))
    for col in range(5):
        ax = axes[row, col]
        ax.axis('off')
        if col < len(sample):
            ax.imshow(Image.open(sample[col]))
        if col == 0:
            ax.set_title(c, loc='left', fontsize=10)
plt.tight_layout()
plt.show()


## 6. Stratified train / val / test split (70 / 15 / 15)

In [ ]:
SPLIT_ROOT = '/content/data/split_sheep'
if os.path.exists(SPLIT_ROOT):
    shutil.rmtree(SPLIT_ROOT)

def list_images(d):
    return [f for f in glob.glob(os.path.join(d, '*')) if f.lower().endswith(IMG_EXTS)]

for c in class_names_unified:
    files_c = list_images(os.path.join(UNIFIED_DIR, c))
    train, temp = train_test_split(files_c, test_size=0.30, random_state=SEED)
    val, test = train_test_split(temp, test_size=0.50, random_state=SEED)
    for split_name, split_files in [('train', train), ('val', val), ('test', test)]:
        out_dir = os.path.join(SPLIT_ROOT, split_name, c)
        os.makedirs(out_dir, exist_ok=True)
        for f in split_files:
            shutil.copy(f, os.path.join(out_dir, os.path.basename(f)))
    print(f"{c:25s} train={len(train):4d}  val={len(val):4d}  test={len(test):4d}")


## 7. Build `tf.data` datasets — 300×300, binary labels

`label_mode='binary'` gives one 0/1 label per image for validation/test (unchanged distribution, so evaluation
stays realistic). **Training uses a balanced sampler instead**: since Orf (572 crops) and Normal (1324 images) are
imbalanced roughly 1:2.3, `tf.data.Dataset.sample_from_datasets` draws from each class with equal probability, so
**every training batch is ~50/50 Orf vs Normal** regardless of the raw file counts. This is a stronger fix than
loss-weighting (`class_weight`) alone: it changes what the model actually *sees* each epoch, not just how heavily
a mistake on the rare class is penalized — which is what was likely producing the "always says Normal" bias.

In [ ]:
IMG_SIZE = (300, 300)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

class_names = sorted(os.listdir(os.path.join(SPLIT_ROOT, 'train')))
DISEASE_INDEX = class_names.index('Contagious_Ecthyma_Orf')
print("Classes (index order):", class_names)
print("Contagious_Ecthyma_Orf is label index:", DISEASE_INDEX)

def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    return img, tf.cast(label, tf.float32)

def class_file_dataset(class_idx):
    pattern = os.path.join(SPLIT_ROOT, 'train', class_names[class_idx], '*')
    files_ds = tf.data.Dataset.list_files(pattern, shuffle=True, seed=SEED).repeat()
    return files_ds.map(lambda p: load_and_preprocess(p, class_idx), num_parallel_calls=AUTOTUNE)

class_datasets = [class_file_dataset(i) for i in range(len(class_names))]
train_ds = tf.data.Dataset.sample_from_datasets(class_datasets, weights=[0.5, 0.5], seed=SEED)
train_ds = train_ds.map(lambda img, label: (img, tf.expand_dims(label, -1)))
train_ds = train_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

def make_eval_dataset(split_name):
    ds = tf.keras.utils.image_dataset_from_directory(
        os.path.join(SPLIT_ROOT, split_name),
        image_size=IMG_SIZE, batch_size=BATCH_SIZE,
        label_mode='binary', shuffle=False, seed=SEED,
    )
    return ds.prefetch(AUTOTUNE)

val_ds  = make_eval_dataset('val')
test_ds = make_eval_dataset('test')

# Since the balanced train_ds repeats forever, model.fit needs an explicit steps_per_epoch. Using roughly
# 2x the larger class's count means the minority class (Orf) gets fully cycled through multiple times per epoch.
train_counts = {c: len(glob.glob(os.path.join(SPLIT_ROOT, 'train', c, '*'))) for c in class_names}
STEPS_PER_EPOCH = int(np.ceil((2 * max(train_counts.values())) / BATCH_SIZE))
print("\nTrain counts (raw, before balancing):", train_counts)
print("Steps per epoch (balanced sampler):", STEPS_PER_EPOCH)


## 8. Why there's no separate "class weights" step here

In the cow and goat notebooks, `class_weight` is what handles imbalance. Here, **Section 7's balanced sampler
already does that job** — every batch is 50/50 by construction, so passing `class_weight` on top of that would
double-correct and likely bias the model the *other* way (toward always predicting Orf). `class_weight` is
therefore intentionally **not** used in the `model.fit()` calls below.

## 9. Build the model — EfficientNetB0, single sigmoid output

In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.35),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
], name='augmentation')

base_model = tf.keras.applications.EfficientNetB0(
    input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet',
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(0.4)(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs, outputs, name='sheep_skin_binary_classifier')
model.summary()


## 10. Train — Phase A: head only (backbone frozen)

In [ ]:
checkpoint_path = 'sheep_skin_best.keras'

callbacks_head = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True),
]

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')],
)

history_head = model.fit(
    train_ds, validation_data=val_ds, epochs=15,
    steps_per_epoch=STEPS_PER_EPOCH, callbacks=callbacks_head,
)


## 11. Train — Phase B: deep fine-tune (70% of backbone unfrozen)

In [ ]:
base_model.trainable = True
n_layers = len(base_model.layers)
for layer in base_model.layers[: int(n_layers * 0.30)]:
    layer.trainable = False

callbacks_finetune = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7),
]

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')],
)

history_finetune = model.fit(
    train_ds, validation_data=val_ds, epochs=30,
    steps_per_epoch=STEPS_PER_EPOCH, callbacks=callbacks_finetune,
)


In [ ]:
def plot_history(h1, h2, metric):
    v1, v2 = h1.history[metric], h2.history[metric]
    vv1, vv2 = h1.history[f'val_{metric}'], h2.history[f'val_{metric}']
    plt.figure(figsize=(9, 4))
    plt.plot(v1 + v2, label='train')
    plt.plot(vv1 + vv2, label='val')
    plt.axvline(len(v1) - 0.5, color='gray', linestyle='--', label='fine-tune starts')
    plt.title(metric); plt.xlabel('epoch'); plt.legend()
    plt.show()

plot_history(history_head, history_finetune, 'accuracy')
plot_history(history_head, history_finetune, 'loss')


## 12. Load the best checkpoint

In [ ]:
model = tf.keras.models.load_model(checkpoint_path)
print(f"Loaded best checkpoint from {checkpoint_path}")


## 13. Calibrate the decision threshold on validation data

The default rule "predict disease if P(disease) ≥ 0.5" assumes the two classes are perfectly balanced and the
model is perfectly calibrated — neither is guaranteed, especially with limited/imbalanced disease data. This finds
the threshold that actually separates the two classes best **on the validation set** (never the test set, to avoid
leaking test information into a decision you then evaluate on that same test set), using Youden's J statistic
(maximizes true-positive rate minus false-positive rate). If the model is systematically under-confident about
Orf, this alone can meaningfully fix a "always says Normal" bias without retraining anything.

In [ ]:
from sklearn.metrics import roc_curve

y_val_true, y_val_prob = [], []
for images, labels in val_ds:
    y_val_true.extend(labels.numpy().flatten().tolist())
    y_val_prob.extend(model.predict(images, verbose=0).flatten().tolist())
y_val_true = np.array(y_val_true)
y_val_prob = np.array(y_val_prob)

fpr, tpr, thresholds = roc_curve(y_val_true, y_val_prob)
youden_j = tpr - fpr
best_idx = np.argmax(youden_j)
BEST_THRESHOLD = float(thresholds[best_idx])
print(f"Default threshold: 0.500")
print(f"Calibrated threshold (Youden's J on validation set): {BEST_THRESHOLD:.3f}")
print(f"(This is a threshold on P(class index 1) = P({class_names[1]}); DISEASE_INDEX={DISEASE_INDEX} is applied "
      f"wherever this threshold is used, same as in predict_sheep_disease() below.)")


## 14. Evaluate on the held-out TEST set, using the calibrated threshold

In [ ]:
y_true, y_prob = [], []
for images, labels in test_ds:
    y_true.extend(labels.numpy().flatten().tolist())
    y_prob.extend(model.predict(images, verbose=0).flatten().tolist())

y_true = np.array(y_true)
y_prob = np.array(y_prob)
y_pred = (y_prob >= BEST_THRESHOLD).astype(int)

test_accuracy = accuracy_score(y_true, y_pred)
test_auc = roc_auc_score(y_true, y_prob)
print(f"TEST ACCURACY (calibrated threshold): {test_accuracy:.1%}")
print(f"TEST AUC (threshold-independent):     {test_auc:.3f}\n")
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

if test_accuracy >= 0.92:
    print("✅ Target of >92% accuracy reached.")
else:
    print("⚠️ Below 92% — see the troubleshooting section at the end of this notebook.")

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title(f'Confusion matrix — test accuracy {test_accuracy:.1%} (threshold={BEST_THRESHOLD:.3f})')
plt.tight_layout()
plt.show()


## 15. Save the model and download it

In [ ]:
model.save('sheep_skin_disease_model.keras')
with open('sheep_class_names.json', 'w') as f:
    json.dump({'class_names': class_names, 'disease_index': DISEASE_INDEX, 'threshold': BEST_THRESHOLD}, f, indent=2)

from google.colab import files
files.download('sheep_skin_disease_model.keras')
files.download('sheep_class_names.json')


## 16. Symptom-based predictor for Sheep (no image needed)

Veterinary knowledge base + synthetic dataset, same approach as the cow and goat notebooks.

In [ ]:
SYMPTOM_KB = {
    'Contagious_Ecthyma_Orf': {
        'core': [
            'scabby pustular lesions around mouth and lips', 'lesions on nose and muzzle',
            'thick crusty scabs', 'lesions on lips can make eating painful',
        ],
        'support': [
            'lesions sometimes on udder or feet', 'young animals affected most severely',
            'lameness if lesions on feet', 'reduced feeding due to mouth pain', 'mild fever',
        ],
    },
    'Normal_Healthy_Skin': {
        'core': [
            'smooth uniform coat', 'no nodules or lesions', 'normal appetite',
            'normal body temperature', 'alert and active',
        ],
        'support': [
            'shiny healthy coat', 'no discharge from eyes or nose', 'normal wool/fleece condition',
        ],
    },
}

ALL_SYMPTOMS = sorted({s for d in SYMPTOM_KB.values() for s in d['core'] + d['support']})
print(f"Total distinct symptoms: {len(ALL_SYMPTOMS)}")
for disease, info in SYMPTOM_KB.items():
    print(f"\n{disease}")
    print("  core   :", ', '.join(info['core']))
    print("  support:", ', '.join(info['support']))


## 17. Generate a synthetic symptom dataset and train the Random Forest

In [ ]:
def sample_symptom_vector(disease, rng):
    info = SYMPTOM_KB[disease]
    vec = {s: 0 for s in ALL_SYMPTOMS}
    for s in info['core']:
        vec[s] = int(rng.random() < 0.85)
    for s in info['support']:
        vec[s] = int(rng.random() < 0.45)
    other_symptoms = [s for s in ALL_SYMPTOMS if s not in info['core'] and s not in info['support']]
    n_noise = rng.choice([0, 0, 0, 1, 1, 2])
    for s in rng.choice(other_symptoms, size=min(n_noise, len(other_symptoms)), replace=False):
        vec[s] = 1
    return vec

rng = np.random.default_rng(SEED)
N_PER_CLASS = 400
rows = []
for disease in SYMPTOM_KB:
    for _ in range(N_PER_CLASS):
        vec = sample_symptom_vector(disease, rng)
        vec['disease'] = disease
        rows.append(vec)

symptom_df = pd.DataFrame(rows).sample(frac=1, random_state=SEED).reset_index(drop=True)

X = symptom_df[ALL_SYMPTOMS].values
y = symptom_df['disease'].values
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)

symptom_model = RandomForestClassifier(
    n_estimators=300, min_samples_leaf=2, random_state=SEED, class_weight='balanced', n_jobs=-1,
)
symptom_model.fit(X_train_s, y_train_s)
symptom_acc = accuracy_score(y_test_s, symptom_model.predict(X_test_s))
print(f"Symptom-model held-out accuracy (on synthetic test data): {symptom_acc:.1%}")

joblib.dump(symptom_model, 'sheep_symptom_model.joblib')
with open('sheep_symptom_list.json', 'w') as f:
    json.dump(ALL_SYMPTOMS, f, indent=2)
files.download('sheep_symptom_model.joblib')
files.download('sheep_symptom_list.json')

def predict_from_symptoms(selected_symptoms):
    vec = np.array([[1 if s in selected_symptoms else 0 for s in ALL_SYMPTOMS]])
    probs = symptom_model.predict_proba(vec)[0]
    classes = symptom_model.classes_
    order = np.argsort(probs)[::-1]
    return [(classes[i], float(probs[i])) for i in order]

print("\nTest:", predict_from_symptoms(SYMPTOM_KB['Contagious_Ecthyma_Orf']['core']))


## 18. Test-Time Augmentation (TTA) for image predictions

Two changes versus the cow/goat notebooks' TTA, specific to this dataset: an extra **tight center-crop view**
(55% zoom) is added, because the training crops are close-up lesion regions cut out of bounding boxes — a real
uploaded photo is usually a wider shot of the face/body, so this view helps bridge that scale gap at inference
time. The final decision also uses the **calibrated threshold** from Section 13 instead of a flat 0.5.

In [ ]:
def tta_views(pil_img):
    img = pil_img.convert('RGB').resize(IMG_SIZE)
    arr = tf.keras.utils.img_to_array(img)
    views = [arr]
    views.append(tf.image.flip_left_right(arr).numpy())
    for zoom in (0.90, 0.75, 0.55):
        h, w = IMG_SIZE
        ch, cw = int(h * zoom), int(w * zoom)
        top, left = (h - ch) // 2, (w - cw) // 2
        cropped = arr[top:top + ch, left:left + cw, :]
        resized = tf.image.resize(cropped, IMG_SIZE).numpy()
        views.append(resized)
        views.append(tf.image.flip_left_right(resized).numpy())
    for delta in (-0.15, 0.15):
        views.append(tf.image.adjust_brightness(arr, delta).numpy())
    return np.stack(views, axis=0)

def predict_sheep_disease(pil_img, use_tta=True):
    if use_tta:
        batch = tta_views(pil_img)
    else:
        img = pil_img.convert('RGB').resize(IMG_SIZE)
        batch = np.expand_dims(tf.keras.utils.img_to_array(img), axis=0)

    probs = model.predict(batch, verbose=0).flatten()
    mean_prob = float(np.mean(probs))  # P(class index 1)

    is_class1 = mean_prob >= BEST_THRESHOLD
    is_diseased = is_class1 if DISEASE_INDEX == 1 else (not is_class1)
    p_disease = mean_prob if DISEASE_INDEX == 1 else (1 - mean_prob)
    confidence = p_disease if is_diseased else (1 - p_disease)
    return is_diseased, confidence

sample_path = glob.glob(os.path.join(SPLIT_ROOT, 'test', 'Contagious_Ecthyma_Orf', '*'))[0]
is_diseased, conf = predict_sheep_disease(Image.open(sample_path))
print(f"Sample known-Orf image -> is_diseased={is_diseased}, confidence={conf:.1%}")


## 19. 🚀 Gradio dashboard — Photo mode (TTA-backed) + Symptom mode

In [ ]:
import gradio as gr

def verdict_markdown(label, confidence):
    pretty = label.replace('_', ' ')
    is_healthy = 'Normal' in label or 'Healthy' in label
    icon = '✅' if is_healthy else '🔴'
    return f"## {icon} Final Diagnosis: **{pretty}**\n\n**Confidence: {confidence:.1%}**"

def gr_predict_image(img):
    if img is None:
        return "Please upload a photo of the sheep's skin."
    is_diseased, confidence = predict_sheep_disease(img, use_tta=True)
    label = 'Contagious_Ecthyma_Orf' if is_diseased else 'Normal_Healthy_Skin'
    return verdict_markdown(label, confidence)

def gr_predict_symptoms(selected):
    if not selected:
        return "Please select at least one symptom.", None
    results = predict_from_symptoms(selected)
    top_label, top_prob = results[0]
    verdict = verdict_markdown(top_label, top_prob)
    breakdown = {label.replace('_', ' '): prob for label, prob in results}
    return verdict, breakdown

with gr.Blocks(title="Sheep Skin Disease Detector") as demo:
    gr.Markdown("# 🐑 Sheep Skin Disease Detector")
    gr.Markdown(
        "Two independent modes — use whichever fits what you have on hand. "
        "This is a decision-support tool, **not** a replacement for a veterinarian."
    )

    with gr.Tab("📷 Diagnose from Photo"):
        with gr.Row():
            with gr.Column():
                img_input = gr.Image(type="pil", label="Upload a skin photo")
                img_btn = gr.Button("Diagnose", variant="primary")
            with gr.Column():
                img_output = gr.Markdown()
        img_btn.click(gr_predict_image, inputs=img_input, outputs=img_output)

    with gr.Tab("📋 Diagnose from Symptoms"):
        gr.Markdown("Tick every symptom you currently observe, then click Diagnose.")
        sym_checkbox = gr.CheckboxGroup(ALL_SYMPTOMS, label="Symptoms")
        sym_btn = gr.Button("Diagnose", variant="primary")
        sym_verdict = gr.Markdown()
        with gr.Accordion("See full breakdown", open=False):
            sym_breakdown = gr.Label(num_top_classes=len(SYMPTOM_KB), label=None)
        sym_btn.click(gr_predict_symptoms, inputs=sym_checkbox, outputs=[sym_verdict, sym_breakdown])

demo.launch(share=True, debug=False)


## 20. If test accuracy is still below 92%, or the bias toward one class persists

1. **Share the class counts from Section 4** and the confusion matrix from Section 14 — that tells us whether this
   is a data-imbalance problem (one class has far fewer images) or a genuine learning problem (model confuses the
   two classes roughly evenly), which need different fixes.
2. **Re-check Section 4's crop counts** — since this dataset is shared with goats, verify visually (Section 5)
   that the "Orf" crops genuinely look like sheep, not goats, if that distinction matters for your report.
3. **Try EfficientNetV2S** as a drop-in replacement for EfficientNetB0 in Section 9.
4. **Increase `pad_frac`** in Section 4 to 0.15–0.20 if lesions are being cropped too tightly.
5. **Check for near-duplicate images across train/test** with `imagehash`.
